# Evaluate DQ Rules - Product table

Evaluates the active rules in `dq.dq_rule` against the `product` table and writes the results to the Lakehouse:

| Table | What is written |
|-------|-----------------|
| `dq.dq_run` | One audit row for this run (Running -> Succeeded / Failed) |
| `dq.dq_record_score` | Weighted score (0-100) and pass/fail status per product |
| `dq.dq_rule_result` | Rule-level result per product (failures only by default) |
| `dq.dq_table_summary` | Totals for the run, for Power BI |

* Run `01_load_product_dq_rules` first.
* Attach this notebook to the Lakehouse that holds the `dq` schema and the product table.
* Scoring: `dq_score = 100 * sum(weight of passed rules) / sum(weight of all rules)`.
  A record passes when `dq_score >= pass_threshold` **and** no critical rule failed.
* A rule whose expression returns `NULL` counts as failed.

In [ ]:
# ---- Config ----------------------------------------------------------------
WORKSPACE_NAME        = "MyWorkspace"
LAKEHOUSE_NAME        = "MyLakehouse"
SCHEMA_NAME           = "dbo"
TABLE_NAME            = "product"
PIPELINE_NAME         = "02_evaluate_product_dq_rules"
STORE_PASSED_RESULTS  = False   # True = also write passed rules to dq_rule_result
EVALUATION_DATE       = None    # None = today; or "YYYY-MM-DD" to pick rule versions for that date

## 1. Load table config and active rules

In [ ]:
import uuid
from datetime import date
from pyspark.sql import functions as F

eval_date = EVALUATION_DATE or date.today().isoformat()

cfg = spark.sql(f'''
    SELECT table_id, business_key_columns, pass_threshold
    FROM dq.dq_source_table
    WHERE workspace_name = '{WORKSPACE_NAME}' AND lakehouse_name = '{LAKEHOUSE_NAME}'
      AND schema_name = '{SCHEMA_NAME}' AND table_name = '{TABLE_NAME}'
      AND is_active = true
''').collect()
if not cfg:
    raise ValueError(f"{SCHEMA_NAME}.{TABLE_NAME} is not registered/active in dq.dq_source_table")

TABLE_ID       = cfg[0].table_id
KEY_COLUMNS    = [c.strip() for c in cfg[0].business_key_columns.split(",") if c.strip()]
PASS_THRESHOLD = float(cfg[0].pass_threshold)

rules = spark.sql(f'''
    SELECT rule_id, rule_name, column_name, rule_expression,
           CAST(weight AS DOUBLE) AS weight, is_critical
    FROM dq.dq_rule
    WHERE table_id = {TABLE_ID} AND is_active = true
      AND effective_from <= DATE'{eval_date}'
      AND (effective_to IS NULL OR effective_to >= DATE'{eval_date}')
    ORDER BY rule_id
''').collect()
if not rules:
    raise ValueError(f"No active rules for table_id {TABLE_ID} on {eval_date}")

TOTAL_WEIGHT = sum(r.weight for r in rules)
print(f"table_id={TABLE_ID}  keys={KEY_COLUMNS}  threshold={PASS_THRESHOLD}  "
      f"rules={len(rules)}  total_weight={TOTAL_WEIGHT}")
display(spark.createDataFrame(rules))

## 2. Start the run

In [ ]:
RUN_ID = str(uuid.uuid4())

try:  # Fabric run context (may be empty when run interactively)
    ctx = notebookutils.runtime.context
    PIPELINE_RUN_ID = ctx.get("currentRunId") or ctx.get("activityId")
except Exception:
    PIPELINE_RUN_ID = None

def _sql_str(v):
    return "NULL" if v is None else "'" + str(v).replace("'", "''") + "'"

spark.sql(f'''
INSERT INTO dq.dq_run
    (run_id, table_id, pipeline_name, pipeline_run_id, run_start_ts, run_end_ts, run_status, records_evaluated)
VALUES
    ('{RUN_ID}', {TABLE_ID}, {_sql_str(PIPELINE_NAME)}, {_sql_str(PIPELINE_RUN_ID)},
     current_timestamp(), NULL, 'Running', NULL)
''')
print(f"run_id = {RUN_ID}")

def finish_run(status, records=None):
    spark.sql(f'''
        UPDATE dq.dq_run
        SET run_status = '{status}', run_end_ts = current_timestamp(),
            records_evaluated = {records if records is not None else 'NULL'}
        WHERE run_id = '{RUN_ID}'
    ''')

## 3. Evaluate rules per record

Each rule becomes a boolean column `_r<rule_id>`. Score, status and rule results are all derived from these columns, and the result is cached so the source is read once.

In [ ]:
try:
    src = spark.table(f"{SCHEMA_NAME}.{TABLE_NAME}")
    missing = [c for c in KEY_COLUMNS if c not in src.columns]
    if missing:
        raise ValueError(f"Business key columns not found in source: {missing}")

    # Business key: JSON text + NULL-safe SHA2-256 hash
    df = (src
        .withColumn("record_key", F.to_json(F.struct(*[F.col(c) for c in KEY_COLUMNS])))
        .withColumn("record_key_hash", F.sha2(F.concat_ws("||",
            *[F.coalesce(F.col(c).cast("string"), F.lit("<NULL>")) for c in KEY_COLUMNS]), 256)))

    # One boolean per rule; NULL result -> failed
    for r in rules:
        df = df.withColumn(f"_r{r.rule_id}", F.coalesce(F.expr(r.rule_expression), F.lit(False)))

    passed_weight = sum(F.when(F.col(f"_r{r.rule_id}"), F.lit(r.weight)).otherwise(F.lit(0.0)) for r in rules)
    rules_failed  = sum(F.when(F.col(f"_r{r.rule_id}"), 0).otherwise(1) for r in rules)
    crit_rules    = [r for r in rules if r.is_critical]
    critical_fail = (F.greatest(*[~F.col(f"_r{r.rule_id}") for r in crit_rules], F.lit(False))
                     if crit_rules else F.lit(False))

    df = (df
        .withColumn("dq_score", F.round(passed_weight / F.lit(TOTAL_WEIGHT) * 100, 2).cast("decimal(5,2)"))
        .withColumn("rules_failed", rules_failed.cast("int"))
        .withColumn("critical_failure", critical_fail)
        .withColumn("dq_status", (F.col("dq_score") >= F.lit(PASS_THRESHOLD)) & ~F.col("critical_failure")))

    # Keep only what the outputs need (incl. checked column values for failed_value)
    rule_cols  = [f"_r{r.rule_id}" for r in rules]
    value_cols = sorted({r.column_name for r in rules if r.column_name and r.column_name in src.columns})
    scored = df.select("record_key", "record_key_hash", "dq_score", "rules_failed",
                       "critical_failure", "dq_status", *rule_cols,
                       *[F.col(c).cast("string").alias(f"_v_{c}") for c in value_cols]).cache()
    RECORDS = scored.count()

    dup = RECORDS - scored.select("record_key_hash").distinct().count()
    if dup:
        print(f"WARNING: {dup} duplicate business keys in source - scores are written per row")
    print(f"Evaluated {RECORDS} records")
except Exception:
    finish_run("Failed")
    raise

## 4. Write record scores (`dq.dq_record_score`)

In [ ]:
try:
    record_scores = (scored
        .withColumn("run_id", F.lit(RUN_ID))
        .withColumn("table_id", F.lit(TABLE_ID).cast("int"))
        .withColumn("record_score_id",
                    F.sha2(F.concat_ws("||", F.lit(RUN_ID), F.lit(str(TABLE_ID)), "record_key_hash"), 256))
        .withColumn("rules_evaluated", F.lit(len(rules)).cast("int"))
        .withColumn("is_current", F.lit(True))
        .withColumn("evaluation_date", F.to_date(F.lit(eval_date)))
        .withColumn("evaluated_ts", F.current_timestamp())
        .select("record_score_id", "run_id", "table_id", "record_key", "record_key_hash",
                "dq_score", "dq_status", "rules_evaluated", "rules_failed", "critical_failure",
                "is_current", "evaluation_date", "evaluated_ts"))

    record_scores.write.format("delta").mode("append").saveAsTable("dq.dq_record_score")

    # Earlier scores for this table are no longer current
    spark.sql(f'''
        UPDATE dq.dq_record_score
        SET is_current = false
        WHERE table_id = {TABLE_ID} AND is_current = true AND run_id <> '{RUN_ID}'
    ''')
    print("dq_record_score written")
except Exception:
    finish_run("Failed")
    raise

## 5. Write rule results (`dq.dq_rule_result`)

In [ ]:
try:
    rule_structs = []
    for r in rules:
        ok = F.col(f"_r{r.rule_id}")
        if r.column_name in value_cols:
            val = F.coalesce(F.col(f"_v_{r.column_name}"), F.lit("<NULL>"))
        else:  # row-level rule
            val = F.lit(None).cast("string")
        rule_structs.append(F.struct(
            F.lit(r.rule_id).cast("int").alias("rule_id"),
            ok.alias("is_passed"),
            F.when(ok, F.lit(r.weight / TOTAL_WEIGHT * 100)).otherwise(F.lit(0.0))
             .cast("decimal(9,4)").alias("rule_score"),
            F.when(~ok, val).alias("failed_value"),
            F.when(~ok, F.lit(f"{r.rule_name} - expected: {r.rule_expression}")).alias("failure_reason"),
        ))

    rule_results = (scored
        .select("record_key_hash", F.explode(F.array(*rule_structs)).alias("rr"))
        .select("record_key_hash", "rr.*"))
    if not STORE_PASSED_RESULTS:
        rule_results = rule_results.where(~F.col("is_passed"))

    (rule_results
        .withColumn("run_id", F.lit(RUN_ID))
        .withColumn("table_id", F.lit(TABLE_ID).cast("int"))
        .withColumn("evaluation_date", F.to_date(F.lit(eval_date)))
        .select("run_id", "table_id", "record_key_hash", "rule_id", "is_passed",
                "rule_score", "failed_value", "failure_reason", "evaluation_date")
        .write.format("delta").mode("append").saveAsTable("dq.dq_rule_result"))
    print("dq_rule_result written")
except Exception:
    finish_run("Failed")
    raise

## 6. Write run summary (`dq.dq_table_summary`) and close the run

In [ ]:
try:
    summary = (scored.agg(
            F.count("*").cast("bigint").alias("total_records"),
            F.coalesce(F.sum(F.col("dq_status").cast("int")), F.lit(0)).cast("bigint").alias("passed_records"),
            F.avg("dq_score").alias("avg_dq_score"))
        .withColumn("failed_records", F.col("total_records") - F.col("passed_records"))
        .withColumn("avg_dq_score", F.round("avg_dq_score", 2).cast("decimal(5,2)"))
        .withColumn("pass_rate_pct", F.when(F.col("total_records") > 0,
            F.round(F.col("passed_records") / F.col("total_records") * 100, 2)).cast("decimal(5,2)"))
        .withColumn("run_id", F.lit(RUN_ID))
        .withColumn("table_id", F.lit(TABLE_ID).cast("int"))
        .withColumn("summary_ts", F.current_timestamp())
        .select("run_id", "table_id", "total_records", "passed_records", "failed_records",
                "avg_dq_score", "pass_rate_pct", "summary_ts"))

    summary.write.format("delta").mode("append").saveAsTable("dq.dq_table_summary")
    finish_run("Succeeded", RECORDS)
    print(f"Run {RUN_ID} succeeded")
except Exception:
    finish_run("Failed")
    raise
finally:
    scored.unpersist()

## 7. Results

In [ ]:
display(spark.sql(f"SELECT * FROM dq.dq_run WHERE run_id = '{RUN_ID}'"))
display(spark.sql(f"SELECT * FROM dq.dq_table_summary WHERE run_id = '{RUN_ID}'"))

In [ ]:
# Failures by rule for this run
display(spark.sql(f'''
    SELECT r.rule_id, r.rule_name, d.dimension_name, r.severity, r.is_critical,
           COUNT(*) AS failed_records
    FROM dq.dq_rule_result rr
    JOIN dq.dq_rule r      ON r.rule_id = rr.rule_id
    JOIN dq.dq_dimension d ON d.dimension_id = r.dimension_id
    WHERE rr.run_id = '{RUN_ID}' AND rr.table_id = {TABLE_ID} AND rr.is_passed = false
    GROUP BY r.rule_id, r.rule_name, d.dimension_name, r.severity, r.is_critical
    ORDER BY failed_records DESC
'''))

In [ ]:
# Lowest-scoring products in this run
display(spark.sql(f'''
    SELECT record_key, dq_score, dq_status, rules_failed, critical_failure
    FROM dq.dq_record_score
    WHERE run_id = '{RUN_ID}' AND table_id = {TABLE_ID}
    ORDER BY dq_score ASC, critical_failure DESC
    LIMIT 50
'''))